# DS 8001 — Lab 6: Search Algorithms

This lab works through the three families of methods from the lecture: **uninformed search** (BFS, DFS, iterative deepening, uniform-cost), **informed search** (greedy best-first, A, A$^*$), and **local search for optimization** (hill climbing, simulated annealing, genetic algorithms).

A recurring theme is that a search strategy is little more than **the order in which the fringe is emptied** — and that this single choice decides whether the method is optimal, how many nodes it expands, and how much memory it needs. In Problems 1, 2, 4 and 5 the algorithms are supplied, and your job is to read them, set up small comparative experiments, and connect what you measure back to the lecture. In Problem 3 you will implement hill climbing yourself.

### How to answer written questions

Whenever you are asked for a written prediction, explanation, or comparative analysis, **insert a new Markdown/text cell immediately after the question and type your answer there**. In Google Colab, use **+ Text**; in Jupyter, insert a new cell and change its type to **Markdown**. Do not put prose answers inside code cells or edit the question itself.

## Setup

Run the next cell first. It imports the packages used throughout the lab, including the `deque` and `heapq` containers that the search algorithms use as a queue and as a priority queue.

In [ ]:
import heapq
import itertools
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Problem 1 — Uninformed search: BFS, DFS and iterative deepening

### **Problem Description**

We will run the uninformed strategies on a small grid-navigation problem, which is a search problem in exactly the sense of the lecture:

- a **state** is a cell `(row, col)`;
- the **initial state** is the cell marked `S` and the **goal state** is the cell marked `G`;
- `succs(s)` returns the cells reachable in one move (up/down/left/right, never through a wall `#`);
- the **cost** of a move is the cost of the cell being entered.

The map uses three kinds of cell: `.` is ordinary terrain (cost 1), `~` is mud (cost 9), and `#` is a wall. Mud lets us switch between two cost models with the same map:

- `grid_unit` — **every move costs 1** (mud is ignored). Used in Part A.
- `grid_costs` — **entering mud costs 9**. Used in Part B and in Problem 2.

### **Provided Code: the search problem**

The next cell defines the map and the `GridProblem` class, which supplies `start`, `is_goal(s)` and `succs(s)` exactly as in the lecture. It also defines `show_path`, which prints a map with a returned path marked by `o` so you can see what a strategy actually did.

In [1]:
# '.' = ordinary terrain (cost 1), '~' = mud, '#' = wall, S = start, G = goal
MAP_ROWS = [
    "S....~~~....G",
    ".....~~~.....",
    ".............",
    "..#########..",
    ".............",
    ".............",
]
MUD_COST = 9

# Successors are generated in this fixed order: East, South, West, North.
MOVES = ((0, 1), (1, 0), (0, -1), (-1, 0))


class GridProblem:
    """A search problem on a grid: states are (row, col) cells."""

    def __init__(self, rows, terrain_costs=True):
        self.grid = [list(r) for r in rows]
        self.n_rows, self.n_cols = len(self.grid), len(self.grid[0])
        self.terrain_costs = terrain_costs
        for r in range(self.n_rows):
            for c in range(self.n_cols):
                if self.grid[r][c] == "S":
                    self.start = (r, c)
                elif self.grid[r][c] == "G":
                    self.goal = (r, c)

    def step_cost(self, s):
        """Cost of entering cell s: 9 in mud when terrain costs are on, else 1."""
        if self.terrain_costs and self.grid[s[0]][s[1]] == "~":
            return MUD_COST
        return 1

    def is_goal(self, s):
        return s == self.goal

    def succs(self, s):
        """States reachable in one move, as (next_state, step_cost) pairs."""
        r, c = s
        out = []
        for dr, dc in MOVES:
            nr, nc = r + dr, c + dc
            # Stay on the board and never enter a wall.
            if 0 <= nr < self.n_rows and 0 <= nc < self.n_cols and self.grid[nr][nc] != "#":
                out.append(((nr, nc), self.step_cost((nr, nc))))
        return out


def show_path(problem, path, title=""):
    """Print the map with the cells of `path` marked by 'o'."""
    out = [row[:] for row in problem.grid]
    for s in path:
        if out[s[0]][s[1]] not in "SG":
            out[s[0]][s[1]] = "o"
    print(title)
    print("\n".join("".join(r) for r in out))


grid_unit = GridProblem(MAP_ROWS, terrain_costs=False)   # every move costs 1
grid_costs = GridProblem(MAP_ROWS, terrain_costs=True)   # mud costs 9

print("\n".join(MAP_ROWS))
print("\nstart:", grid_unit.start, " goal:", grid_unit.goal)

S....~~~....G
.....~~~.....
.............
..#########..
.............
.............

start: (0, 0)  goal: (0, 12)


### **Provided Algorithms: one search function, three fringes**

The lecture stressed that BFS and DFS differ *only* in the container used for the fringe. The next cell makes that literal: `search` is a single function whose `strategy` argument selects

- `"bfs"` — a **queue** (FIFO), so the shallowest generated node leaves first;
- `"dfs"` — a **stack** (LIFO), so the deepest generated node leaves first;
- `"best"` — a **priority queue** ordered by a `priority(g, state)` function you supply. This is the container used by uniform-cost search in Part B and by the informed strategies in Problem 2.

Two details follow the lecture. The **goal test fires when a node is removed from the fringe**, not when it is generated — this is what lets uniform-cost search and A$^*$ be optimal. And because the grid is a graph rather than a tree, `search` keeps a **CLOSED** set of already-expanded states and discards any state it meets again.

Each call returns a dictionary with the solution `path`, its `steps` and `cost`, the number of nodes `expanded`, and `max_stored`, the largest number of states held at one time (fringe + CLOSED).

In [2]:
def search(problem, strategy, priority=None):
    """Graph search with a pluggable fringe.

    strategy = "bfs"  -> queue (FIFO)
             = "dfs"  -> stack (LIFO)
             = "best" -> priority queue ordered by priority(g, state)
    """
    start = problem.start
    tie = itertools.count()  # breaks priority ties in insertion order

    if strategy == "bfs":
        fringe = deque([(start, [start], 0)])
        pop = fringe.popleft          # take from the front: shallowest first
        push = fringe.append
    elif strategy == "dfs":
        fringe = [(start, [start], 0)]
        pop = fringe.pop              # take from the back: deepest first
        push = fringe.append
    elif strategy == "best":
        fringe = []
        heapq.heappush(fringe, (priority(0, start), next(tie), (start, [start], 0)))
        pop = lambda: heapq.heappop(fringe)[2]
        push = lambda item: heapq.heappush(
            fringe, (priority(item[2], item[0]), next(tie), item))
    else:
        raise ValueError(strategy)

    closed = set()
    expanded = 0
    max_stored = 1

    while fringe:
        state, path, g = pop()

        # Graph search: this state has already been expanded, so throw it away.
        if state in closed:
            continue

        # The goal test fires on removal from the fringe, not on generation.
        if problem.is_goal(state):
            return dict(path=path, steps=len(path) - 1, cost=g,
                        expanded=expanded, max_stored=max_stored)

        closed.add(state)
        expanded += 1

        for nxt, step in problem.succs(state):
            if nxt not in closed:
                push((nxt, path + [nxt], g + step))

        max_stored = max(max_stored, len(fringe) + len(closed))

    return None  # failure: no goal is reachable

Iterative deepening needs a **depth-limited** DFS, so it is written separately. `depth_limited` runs a recursive DFS that refuses to go deeper than `limit`, and `iterative_deepening` calls it with limits $0, 1, 2, \dots$ until a goal is found.

Note the difference in memory from `search`: `depth_limited` keeps no CLOSED set at all. It only avoids repeating a state that is already on the **current path**, so the states it stores at any moment are just that one path. Its `max_stored` is therefore the length of the longest path it held, and its `expanded` count is summed over *all* the deepening iterations.

In [3]:
def depth_limited(problem, limit):
    """Recursive DFS that never goes deeper than `limit`."""
    stats = {"expanded": 0, "max_stored": 1}

    def recurse(state, path, g, depth):
        stats["max_stored"] = max(stats["max_stored"], len(path))

        if problem.is_goal(state):
            return dict(path=path, steps=len(path) - 1, cost=g)

        # Depth limit reached: back up without expanding this node.
        if depth == limit:
            return None

        stats["expanded"] += 1
        for nxt, step in problem.succs(state):
            if nxt in path:       # only the current path is remembered
                continue
            found = recurse(nxt, path + [nxt], g + step, depth + 1)
            if found:
                return found
        return None

    return recurse(problem.start, [problem.start], 0, 0), stats


def iterative_deepening(problem, max_limit=40):
    """Depth-limited DFS with limits 0, 1, 2, ... until a goal is found."""
    total_expanded = 0
    max_stored = 1

    for limit in range(max_limit + 1):
        result, stats = depth_limited(problem, limit)
        total_expanded += stats["expanded"]      # work is never reused
        max_stored = max(max_stored, stats["max_stored"])
        if result:
            result = dict(result)
            result.update(expanded=total_expanded, max_stored=max_stored,
                          depth_limit=limit)
            return result

    return None

### **Task Description**

#### Part A — Three strategies on a unit-cost map

Use `grid_unit`, where every move costs 1.

1. Run BFS, DFS and iterative deepening on `grid_unit`.
2. Collect the results in a table with one row per strategy and columns for **path steps**, **path cost**, **nodes expanded** and **max nodes stored**.
3. Use `show_path` to display the BFS path and the DFS path.

**Written prediction:** before running the cell, add a new Markdown/text cell immediately below this question and predict, in 1–2 sentences, which of the three strategies will return the shortest path and which will store the fewest nodes.

In [4]:
runs = {
    "BFS": lambda: search(grid_unit, "bfs"),
    "DFS": lambda: search(grid_unit, "dfs"),
    "IDS": lambda: iterative_deepening(grid_unit),
}

results_a = {}
rows = []

for name, run in runs.items():
    # TODO: call run(), store the result in results_a[name], and append a dict to
    #       `rows` with the strategy name, path steps, path cost, nodes expanded
    #       and max nodes stored.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: use show_path(grid_unit, results_a["BFS"]["path"], "BFS path:") and the
#       same for DFS to see the two routes.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Which strategies returned a shortest path and which did not? Iterative deepening re-runs its depth-limited search from scratch at every limit — what did that cost it in the table, and what did it buy in return? Write 2–3 sentences._

#### Part B — When costs are not all equal

Now switch to `grid_costs`, where entering a mud cell `~` costs 9 instead of 1. The map is unchanged; only the cost model differs.

1. Run BFS on `grid_costs`, and run uniform-cost search by calling `search(grid_costs, "best", priority=...)` with a priority function that orders the fringe by the path cost `g`.
2. Report **path steps**, **path cost** and **nodes expanded** for both in a small table.
3. Use `show_path` to display both routes.

In [5]:
def priority_g(g, state):
    """Uniform-cost search: order the fringe by the path cost g(s)."""
    # TODO: return the value that uniform-cost search should sort the fringe by.
    pass


rows = []
results_b = {}

for name, kwargs in [("BFS", dict(strategy="bfs")),
                     ("UCS", dict(strategy="best", priority=priority_g))]:
    # TODO: run search(grid_costs, **kwargs), store it in results_b[name], and
    #       append a dict with the strategy name, path steps, path cost and
    #       nodes expanded.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: show_path both routes on grid_costs.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. BFS returned the same path as in Part A, but it is no longer the answer we want. Explain in 2–3 sentences what BFS is optimal with respect to, and what changed about the problem that makes uniform-cost search the right choice._

## Problem 2 — Informed search: greedy, A, and A$^*$

### **Problem Description**

Uninformed search knows only $g(s)$, the cost of the path already travelled. Informed search adds a **heuristic** $h(s)$ estimating the cost still to go. The lecture built up to A$^*$ in three attempts, and all three are the *same* algorithm as uniform-cost search with a different fringe ordering:

| strategy | fringe priority |
|---|---|
| uniform-cost | $g(s)$ |
| greedy best-first | $h(s)$ |
| A / A$^*$ | $f(s) = g(s) + h(s)$ |

That is why no new search code is needed here: you will reuse `search(grid_costs, "best", priority=...)` from Problem 1 and change only the priority function.

### **Provided Code: the heuristic**

The heuristic is the Manhattan distance to the goal. It is admissible, by the lecture's relaxation argument: it is the exact cost of a **relaxed** problem in which there are no walls and no mud, so it can never over-estimate the true cost $h^*(s)$ in the real problem.

In [6]:
def manhattan(state, goal):
    """Manhattan distance |row difference| + |column difference| to the goal.

    This is the exact cost of the relaxed problem with no walls and no mud,
    so it never over-estimates the true cost-to-go: the heuristic is admissible.
    """
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


print("h(start) =", manhattan(grid_costs.start, grid_costs.goal))

h(start) = 12


### **Task Description**

#### Part A — Three priorities, one search function

1. Write the three priority functions below. Each takes the path cost `g` and the `state`, and returns the value the fringe should be sorted by. Use `manhattan(state, grid_costs.goal)` for $h$.
2. Run all three on `grid_costs` and tabulate **path cost**, **path steps** and **nodes expanded**.
3. Display the greedy path and the A$^*$ path with `show_path`.

In [7]:
def priority_ucs(g, state):
    """Uniform-cost search: priority g(s)."""
    # TODO
    pass


def priority_greedy(g, state):
    """Greedy best-first search: priority h(s)."""
    # TODO
    pass


def priority_astar(g, state):
    """A* search: priority f(s) = g(s) + h(s)."""
    # TODO
    pass


rows = []
results_2 = {}

for name, pri in [("UCS (g)", priority_ucs),
                  ("Greedy (h)", priority_greedy),
                  ("A* (g+h)", priority_astar)]:
    # TODO: run search(grid_costs, "best", priority=pri), store it in
    #       results_2[name], and append path cost, path steps, nodes expanded.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: show_path the greedy route and the A* route.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Greedy best-first expanded the fewest nodes of the three. Why is that not enough to make it the best strategy here, and what does A$^*$ get from adding $g(s)$ back into the priority? Write 2–3 sentences._

#### Part B — What admissibility actually buys

The lecture's requirement for A$^*$ is that $h(s) \le h^*(s)$ for every state. Let us break that requirement deliberately and see what happens: scale the heuristic by a factor $w \ge 1$, so $h_w(s) = w \cdot \texttt{manhattan}(s)$. For $w = 1$ this is the admissible heuristic from Part A; for larger $w$ it over-estimates.

1. Complete `make_weighted_priority(w)` so that it returns an A-search priority $g(s) + w \cdot h(s)$.
2. Get $h^*(\text{start})$, the true cheapest cost from the start, from the uniform-cost run.
3. For each $w$ in the supplied list, run the search and record $h_w(\text{start})$, whether $h_w(\text{start}) \le h^*(\text{start})$, the **path cost** returned, and the **nodes expanded**.
4. Make a figure with two panels: nodes expanded against $w$, and path cost against $w$ with a horizontal line at the optimal cost.

Note that checking the start state only ever *disproves* admissibility: one over-estimate is enough to show $h_w$ is inadmissible, but passing the check at a single state does not prove a heuristic is admissible everywhere.

In [8]:
def make_weighted_priority(w):
    """Return an A-search priority function using the heuristic w * manhattan."""
    def priority(g, state):
        # TODO: return g + w * (Manhattan distance from state to grid_costs.goal).
        pass
    return priority


weights = [1, 2, 3, 5, 6, 8, 10]

# TODO: set h_star_start to the true cheapest cost from the start. You already
#       computed it: it is the path cost returned by uniform-cost search.
h_star_start = None

rows = []
for w in weights:
    # TODO: run search(grid_costs, "best", priority=make_weighted_priority(w))
    #       and append w, h_w(start), whether it stays at or below h*(start),
    #       the path cost and the nodes expanded.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: make a figure with two panels (plt.subplots(1, 2, figsize=(11, 4))):
#       left  = nodes expanded vs w
#       right = path cost vs w, with plt.axhline at h_star_start

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. What happens to the number of nodes expanded as $w$ grows, and what happens to the cost of the path returned? Some inadmissible weights still returned an optimal path — does that make them safe to use? Write 2–3 sentences._

## Problem 3 — Hill climbing for $N$-queens

### **Problem Description**

We now move from path-finding to **optimization**. There is no path to recover: every state $s$ has a score $f(s)$, and we only want a state with a good score.

The problem is the lecture's $N$-queens, in the one-queen-per-column encoding: a state is a tuple of 8 numbers, where entry $i$ is the row of the queen in column $i$. Because each column holds exactly one queen, column conflicts are impossible by construction, and

$$f(s) = \text{number of pairs of queens attacking each other (same row or same diagonal)},$$

which we want to **minimize** to 0. The neighborhood is the lecture's: move one queen vertically within its own column, giving $8 \times 7 = 56$ neighbors.

### **Provided Code: the objective and the neighborhood**

The next cell supplies `conflicts` (the score $f$), `neighbors` (the move set) and `random_state`. It also keeps a counter of how many times $f$ has been evaluated: `reset_evals()` zeroes it and `n_evals()` reads it. Counting evaluations rather than seconds gives us a fair way to compare methods later, because it does not depend on how fast any one implementation happens to be.

In [9]:
N_QUEENS = 8
MAX_PAIRS = N_QUEENS * (N_QUEENS - 1) // 2   # 28 pairs of queens in total

_EVALS = {"n": 0}


def reset_evals():
    """Set the objective-evaluation counter back to zero."""
    _EVALS["n"] = 0


def n_evals():
    """How many times conflicts() has been called since the last reset."""
    return _EVALS["n"]


def conflicts(state):
    """f(s): number of attacking pairs. 0 means the board is solved."""
    _EVALS["n"] += 1
    n = len(state)
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            # Same row, or same diagonal (column gap equals row gap).
            if state[i] == state[j] or abs(state[i] - state[j]) == j - i:
                total += 1
    return total


def neighbors(state):
    """Move set: move one queen to a different row within its own column."""
    out = []
    for col in range(len(state)):
        for row in range(len(state)):
            if row != state[col]:
                t = list(state)
                t[col] = row
                out.append(tuple(t))
    return out


def random_state(rng, n=N_QUEENS):
    """A random board: one queen per column, at a random row."""
    return tuple(int(x) for x in rng.integers(0, n, size=n))


demo_rng = np.random.default_rng(0)
demo = random_state(demo_rng)
print("a random state:", demo)
print("conflicts:", conflicts(demo), " number of neighbors:", len(neighbors(demo)))

NameError: name 'np' is not defined

### **Task Description**

#### Part A — Implement hill climbing

This is the one algorithm you write yourself. Follow the lecture's pseudocode, written here for **minimization** since we are reducing conflicts:

```text
HillClimb(s)
    repeat
        t = the neighbor of s with the smallest f
        if f(t) >= f(s)
            return s              # no neighbor is strictly better: stop
        s = t
```

Complete `hill_climb` so that it returns the triple `(final_state, final_conflicts, n_steps)`, where `n_steps` counts the moves actually taken. `min(neighbors(s), key=conflicts)` is a convenient way to find the best neighbor.

In [ ]:
def hill_climb(state):
    """Steepest-descent hill climbing. Returns (state, conflicts, n_steps)."""
    current = state
    steps = 0

    while True:
        # TODO: find the neighbor of `current` with the smallest number of conflicts.

        # TODO: if that neighbor is not strictly better than `current`, we are at a
        #       local optimum: return (current, conflicts(current), steps).

        # TODO: otherwise move to that neighbor and count the step.
        raise NotImplementedError("Complete hill_climb, then delete this line.")


# A quick check on one random start.
check_rng = np.random.default_rng(1)
start_state = random_state(check_rng)
end_state, end_conflicts, n_steps = hill_climb(start_state)
print("start:", start_state, "with", conflicts(start_state), "conflicts")
print("end:  ", end_state, "with", end_conflicts, "conflicts after", n_steps, "steps")

#### Part B — How often does hill climbing get stuck, and how many restarts fix it?

The lecture claims that if one hill-climbing run succeeds with probability $p$, then repeated hill climbing needs about $1/p$ restarts on average. Let us measure both quantities.

1. Run `hill_climb` from 200 random starts and keep the number of conflicts each run ended at.
2. Report $p$, the fraction of runs that reached 0 conflicts, together with $1/p$.
3. Bar-plot how often the runs ended at 0, 1, 2, ... conflicts.
4. Then simulate repeated hill climbing: 200 times over, keep restarting from a fresh random state until a run reaches 0 conflicts, and record how many attempts that took. Compare the mean number of restarts with $1/p$.

In [10]:
rng = np.random.default_rng(8001)
N_STARTS = 200

reset_evals()
final_conflicts = []
for _ in range(N_STARTS):
    # TODO: run hill_climb from a random start and append its final conflict count.
    pass

# TODO: compute p, the fraction of runs that ended at 0 conflicts.
#       Print p, 1/p, and n_evals() / N_STARTS (the mean cost of one run).
p = None

# TODO: bar-plot how many runs ended at each number of conflicts.
#       pd.Series(final_conflicts).value_counts().sort_index() is a convenient start.

# --- repeated hill climbing: how many restarts until a success? ---
rng2 = np.random.default_rng(4242)
restarts_needed = []
for _ in range(200):
    # TODO: restart hill_climb from a fresh random state, counting attempts,
    #       until one reaches 0 conflicts. Append that count.
    pass

# TODO: print the mean of restarts_needed next to 1/p.

NameError: name 'np' is not defined

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Most runs stopped at a state with only one or two conflicts rather than at a badly broken board — what does that tell you about the landscape hill climbing is searching? And how well did the mean number of restarts match $1/p$? Write 2–3 sentences._

## Problem 4 — Simulated annealing and the cooling schedule

### **Problem Description**

Hill climbing gets stuck because it will only ever move to a strictly better neighbor. Simulated annealing relaxes exactly that: it picks a **random** neighbor, always accepts an improvement, and accepts a worse state with the Metropolis probability

$$\exp\!\left(-\frac{|f(s) - f(t)|}{\textit{Temp}}\right),$$

where the temperature is multiplied by a fixed `cooling` rate at every iteration. At high temperature almost anything is accepted (a random walk); at low temperature only improvements survive (first-choice hill climbing). The lecture warns that the schedule matters: cool too fast and the search freezes in a local optimum, cool too slowly and it never settles.

### **Provided Algorithm**

`simulated_annealing` below is the lecture's algorithm, on the same $N$-queens objective and neighborhood as Problem 3. It returns the **final** state, as written in the lecture, not the best state ever visited. `iters_to_cool` is a small helper reporting how many iterations a given schedule needs before the temperature drops below 0.01 — a useful way to read a cooling rate against the iteration budget.

In [ ]:
def simulated_annealing(state, rng, T0=5.0, cooling=0.99, n_iters=2000):
    """Simulated annealing on N-queens. Returns (final_state, final_conflicts)."""
    s = state
    f_s = conflicts(s)
    T = T0

    for _ in range(n_iters):
        if f_s == 0:                      # solved: nothing left to improve
            break

        # Pick ONE random neighbor rather than the best one.
        nbrs = neighbors(s)
        t = nbrs[int(rng.integers(len(nbrs)))]
        f_t = conflicts(t)

        # Always accept an improvement; otherwise accept with the Metropolis
        # probability, which shrinks as the move gets worse and as T falls.
        if f_t < f_s or (T > 1e-12 and rng.random() < math.exp(-(f_t - f_s) / T)):
            s, f_s = t, f_t

        T *= cooling                      # anneal

    return s, f_s


def iters_to_cool(cooling, T0=5.0, target=0.01):
    """Iterations needed before the temperature falls below `target`."""
    return math.log(target / T0) / math.log(cooling)


print("iterations to cool below 0.01, for a 2000-iteration budget:")
for c in [0.5, 0.99, 0.9999]:
    print(f"  cooling={c}: {iters_to_cool(c):.0f}")

### **Task Description**

Compare cooling schedules under a fixed budget of 2000 iterations.

1. For each cooling rate in `cooling_rates`, run simulated annealing `N_RUNS` times from random starts. Re-create the random generator with the same seed at the start of each schedule so that every schedule sees the same starting boards.
2. Record the cooling rate, `iters_to_cool(cooling)`, the **success rate** (fraction of runs ending at 0 conflicts), and the **mean final conflicts**.
3. Display the table and plot success rate against the cooling rate on a logarithmic x-axis.

**Written prediction:** before running the cell, add a Markdown/text cell below this question and predict in one sentence whether the fastest cooling rate, the slowest, or something in between will do best.

In [ ]:
cooling_rates = [0.5, 0.9, 0.95, 0.99, 0.999, 0.9999]
N_RUNS = 60
N_ITERS = 2000

rows = []
for cooling in cooling_rates:
    rng = np.random.default_rng(7)   # same starting boards for every schedule
    reset_evals()

    # TODO: run simulated_annealing N_RUNS times, each from random_state(rng),
    #       passing rng, cooling=cooling and n_iters=N_ITERS. Collect the final
    #       conflict counts.

    # TODO: append a dict with the cooling rate, iters_to_cool(cooling),
    #       the success rate and the mean final conflicts.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: plot success rate against cooling rate with plt.xscale("log").

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. Using the "iters to cool" column, explain what goes wrong at the fastest cooling rate and what goes wrong at the slowest. How does the best schedule compare with the plain hill-climbing success rate $p$ from Problem 3? Write 2–3 sentences._

## Problem 5 — Genetic algorithms, and a fair comparison

### **Problem Description**

A genetic algorithm keeps a whole **population** of states instead of a single one, and generates new states by **crossover** and **mutation**, with parents chosen by **proportional selection**. Following the lecture, a state is the same 8-digit string as before and fitness is the number of **non**-attacking pairs,

$$\textit{fitness}(s) = 28 - \textit{conflicts}(s),$$

so 28 is a solved board and we are now maximizing.

### **Provided Algorithms**

`genetic_algorithm` is the lecture's variation: reproduction probabilities $p_i = f(s_i) / \sum_j f(s_j)$, one-point crossover at a random cut, mutation of each position with a small probability, and the new generation replacing the old. Its `elitism` switch is one of the variations listed at the end of the lecture — when on, the best individual of the current generation is copied into the next one unchanged.

`repeated_hill_climb` is the random-restart method from Problem 3, rewritten to stop once it has used a given number of objective evaluations. That lets us give every method the **same budget** in Part B, which is the only fair way to compare methods that do very different amounts of work per step.

In [ ]:
def fitness(state):
    """Number of NON-attacking pairs: 28 is a solved board (maximize)."""
    return MAX_PAIRS - conflicts(state)


def genetic_algorithm(rng, pop_size=40, n_generations=120, mutation_rate=0.05,
                      elitism=False, n=N_QUEENS):
    """Returns (best_state, best_fitness, history of the best fitness per generation)."""
    pop = [random_state(rng, n) for _ in range(pop_size)]
    fits = [fitness(s) for s in pop]
    history = [max(fits)]

    for _ in range(n_generations):
        if max(fits) == MAX_PAIRS:        # solved
            break

        # Proportional selection: fitter individuals are likelier, not certain.
        probs = np.array(fits, dtype=float)
        probs = probs / probs.sum()

        children = []
        for _ in range(pop_size // 2):
            i, j = rng.choice(pop_size, size=2, replace=False, p=probs)
            p1, p2 = pop[i], pop[j]
            cut = int(rng.integers(1, n))          # one-point crossover
            children.append(p1[:cut] + p2[cut:])
            children.append(p2[:cut] + p1[cut:])

        # Mutate every position of every child with a small probability.
        for k in range(len(children)):
            child = list(children[k])
            for pos in range(n):
                if rng.random() < mutation_rate:
                    child[pos] = int(rng.integers(n))
            children[k] = tuple(child)

        # Elitism: carry the current best individual over unchanged.
        if elitism:
            children[0] = pop[int(np.argmax(fits))]

        pop = children
        fits = [fitness(s) for s in pop]
        history.append(max(fits))

    return pop[int(np.argmax(fits))], max(fits), history


def repeated_hill_climb(rng, budget):
    """Random-restart hill climbing, stopped after `budget` objective evaluations.

    Returns (best_conflicts, n_restarts). Uses the hill_climb you wrote in
    Problem 3, so the number of restarts that fit in the budget depends on
    how many evaluations your implementation spends per run.
    """
    best_f, n_restarts = None, 0
    start = n_evals()

    while n_evals() - start < budget:
        _, f, _ = hill_climb(random_state(rng))
        n_restarts += 1
        if best_f is None or f < best_f:
            best_f = f
        if best_f == 0:                   # solved: stop early
            break

    return best_f, n_restarts

### **Task Description**

#### Part A — Does elitism matter?

1. Run the genetic algorithm `RUNS` times with `elitism=False` and `RUNS` times with `elitism=True`, re-creating the generator with the same seed before each setting so both see the same starting populations.
2. Tabulate the **success rate** (fraction of runs reaching fitness 28), the **mean best fitness** and the **mean evaluations per run**.
3. Plot the `history` of one run of each setting: best fitness in the population against generation.

In [ ]:
POP_SIZE, N_GENERATIONS, MUTATION_RATE = 40, 120, 0.05
RUNS = 40

rows = []
histories = {}

for elitism in [False, True]:
    rng = np.random.default_rng(11)   # same starting populations both times
    reset_evals()

    # TODO: run genetic_algorithm RUNS times with pop_size=POP_SIZE,
    #       n_generations=N_GENERATIONS, mutation_rate=MUTATION_RATE and this
    #       elitism setting. Keep the best fitness of each run, and store the
    #       history of the FIRST run in histories[elitism].

    # TODO: append a dict with elitism, the success rate (best fitness == MAX_PAIRS),
    #       the mean best fitness and n_evals() / RUNS.
    pass

# TODO: build a DataFrame from `rows` and display it.

# TODO: plot histories[False] and histories[True] against generation on one figure.

#### Part B — An equal-budget comparison

All three local-search methods can now be given the same budget of objective evaluations. A genetic algorithm run costs at most `POP_SIZE * N_GENERATIONS` evaluations, so use that as the budget.

1. Run each of the three methods `RUNS` times with that budget:
   - `repeated_hill_climb(rng, BUDGET)`;
   - `simulated_annealing(..., cooling=0.99, n_iters=BUDGET)`, the best schedule from Problem 4;
   - `genetic_algorithm(..., elitism=True, n_generations=BUDGET // POP_SIZE)`.
2. Record the **success rate** and the **mean evaluations actually used** for each, and display the table.

Remember to reset the generator and the evaluation counter before each method.

In [ ]:
BUDGET = POP_SIZE * N_GENERATIONS
print("budget =", BUDGET, "objective evaluations per run")

rows = []

# --- repeated hill climbing ---
rng = np.random.default_rng(11)
reset_evals()
# TODO: run repeated_hill_climb(rng, BUDGET) RUNS times, and append a row with
#       the method name, the success rate (best conflicts == 0) and n_evals()/RUNS.

# --- simulated annealing ---
rng = np.random.default_rng(11)
reset_evals()
# TODO: run simulated_annealing from random starts RUNS times with cooling=0.99
#       and n_iters=BUDGET, and append the same three quantities.

# --- genetic algorithm ---
rng = np.random.default_rng(11)
reset_evals()
# TODO: run genetic_algorithm RUNS times with pop_size=POP_SIZE, elitism=True,
#       mutation_rate=MUTATION_RATE and n_generations=BUDGET // POP_SIZE.
#       Success here means best fitness == MAX_PAIRS.

# TODO: build a DataFrame from `rows` and display it.

**Your comparative analysis:**  
_Add a new Markdown/text cell immediately below this question. What does elitism change about the generational replacement step, and why does that matter given how parents are selected? Then look at the equal-budget table: the lecture ends both the annealing and the genetic-algorithm sections with "Try hill-climbing with random restarts first!" — does your table support that advice? Write 3–4 sentences._

## Final takeaway

Across these five problems, one idea did most of the work: **a search strategy is a rule for choosing what to look at next**, and almost everything else follows from it.

- In Problem 1 the only difference between BFS and DFS was a queue versus a stack, and that alone decided whether the path returned was the shallowest. Iterative deepening bought BFS-quality answers with DFS-sized memory by paying for it in repeated work.
- In Problem 2 uniform-cost, greedy best-first and A$^*$ were literally the same function with the priority changed from $g$ to $h$ to $g + h$. Admissibility was the condition that turned a fast method into a fast *and* optimal one — and the weighted runs showed that once it is broken, a correct answer is luck rather than a guarantee.
- In Problems 3–5 there was no path to find, only a score to improve. Hill climbing stopped at states one move away from a solution because a local optimum is indistinguishable from a global one when all you can see is the neighborhood. Random restarts and simulated annealing both escaped that trap, and at an equal budget the simplest of the three methods was the strongest.

The useful question is therefore not *"which algorithm is best?"* but *"what does this strategy order its choices by, what guarantee does that ordering give me, and what does it cost in nodes, memory, or evaluations?"* Two choices we did not vary matter just as much: the heuristic in Problem 2 and the neighborhood in Problems 3–5 are, as the lecture put it, where the real ingenuity lies.